<a href="https://colab.research.google.com/github/aiwithpavansama/GenerativeandAgenticAI/blob/main/Automating_the_Resume_Screening_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install -q sentence-transformers pymupdf python-docx scikit-learn pandas

In [41]:
import os, re, glob
import fitz                      # PyMuPDF, reads PDFs
import numpy as np
import pandas as pd
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!pip install pypdf sentence-transformers scikit-learn

import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FOLDER = '/content/drive/MyDrive/20_Job_Descriptions (2).pdf'

def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

In [43]:
JD_PDF_PATH   = "/content/drive/MyDrive/20_Job_Descriptions (2).pdf"  # the single PDF holding 20 JDs
RESUME_FOLDER = "/content/drive/MyDrive/Education"                   # folder with all resumes
TOP_K         = 10                                                # how many resumes per JD

EXPECTED_JDS  = 20
MODEL_NAME    = "all-MiniLM-L6-v2"   # fast & free. Use "all-mpnet-base-v2" for higher quality (slower)

In [ ]:
def extract_pdf_pages(path):
    doc = fitz.open(path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return pages

def load_job_descriptions(pdf_path, expected=20):
    pages = extract_pdf_pages(pdf_path)
    full_text = "\n".join(pages)

    # Strategy A: assume one job description per page
    by_page = [p.strip() for p in pages if p.strip()]
    if len(by_page) == expected:
        return by_page

    # Strategy B: split the whole text on common JD headings
    # -> tweak this regex to match how YOUR job descriptions start
    pattern = r"(?i)(?=job\s*description\s*\d*|job\s*title\s*[:\-]|position\s*[:\-]|role\s*[:\-])"
    by_pattern = [p.strip() for p in re.split(pattern, full_text) if p.strip()]
    if len(by_pattern) >= expected:
        return by_pattern[:expected]

    print(f"WARNING: page-split gave {len(by_page)}, pattern-split gave {len(by_pattern)}, "
          f"expected {expected}. Inspect the preview below and adjust the splitting logic.")
    return by_page if by_page else by_pattern

jd_texts = load_job_descriptions(JD_PDF_PATH, EXPECTED_JDS)
print(f"Loaded {len(jd_texts)} job descriptions\n")

# Preview the first line of each JD so you can confirm the split is correct
for i, t in enumerate(jd_texts, 1):
    first_line = t.strip().splitlines()[0] if t.strip() else "(empty)"
    print(f"{i:2d}. {first_line[:80]}")

In [ ]:
def extract_resume_text(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".pdf":
            doc = fitz.open(path)
            text = "\n".join(page.get_text() for page in doc)
            doc.close()
            return text
        elif ext == ".docx":
            return "\n".join(p.text for p in Document(path).paragraphs)
        elif ext == ".txt":
            with open(path, "r", errors="ignore") as f:
                return f.read()
    except Exception as e:
        print(f"Could not read {path}: {e}")
    return ""

resume_paths = []
for ext in ("*.pdf", "*.docx", "*.txt"):
    resume_paths += glob.glob(os.path.join(RESUME_FOLDER, "**", ext), recursive=True)

resume_texts, resume_names = [], []
for p in sorted(resume_paths):
    txt = extract_resume_text(p)
    if txt.strip():
        resume_texts.append(txt)
        resume_names.append(os.path.basename(p))

print(f"Loaded {len(resume_texts)} resumes")

In [ ]:
model = SentenceTransformer(MODEL_NAME)   # uses GPU automatically if the Colab runtime has one

jd_emb = model.encode(jd_texts, convert_to_numpy=True,
                      normalize_embeddings=True, show_progress_bar=True)
res_emb = model.encode(resume_texts, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=True)

print("JD embeddings:", jd_emb.shape, " Resume embeddings:", res_emb.shape)

In [ ]:
sim = cosine_similarity(jd_emb, res_emb)   # shape: (num_jds, num_resumes)

rows = []
k = min(TOP_K, len(resume_names))
for i in range(len(jd_texts)):
    top_idx = np.argsort(sim[i])[::-1][:k]
    for rank, idx in enumerate(top_idx, 1):
        rows.append({
            "JD_number": i + 1,
            "rank":      rank,
            "resume":    resume_names[idx],
            "score":     round(float(sim[i][idx]), 4),
        })

results = pd.DataFrame(rows)
results.head(TOP_K)

In [ ]:
for i in range(len(jd_texts)):
    print(f"\n=== Job Description {i+1} ===")
    block = results[results.JD_number == i + 1]
    for _, r in block.iterrows():
        print(f"  {int(r['rank']):2d}. {r['resume']:<45} score={r['score']}")

In [ ]:
out_path = "/content/drive/MyDrive/csv/jd_resume_matches.csv"
results.to_csv(out_path, index=False)
print("Saved to", out_path)

from google.colab import files
files.download(out_path)